In [0]:
import requests
import pandas as pd

API_KEY = "b35b22ee880a408f98576760563204e6"

headers = {
    "X-Auth-Token": API_KEY
}

url = "https://api.football-data.org/v4/competitions/BSA/matches"
params = {
    "season": 2026
}

response = requests.get(url, headers=headers, params=params)
response.raise_for_status()

data = response.json()

In [0]:
matches_pd = pd.json_normalize(data["matches"])
matches_pd.head()

In [0]:
matches_spark = spark.createDataFrame(matches_pd)

def clean_column_names(df):
    new_cols = []
    for c in df.columns:
        clean = (
            c.replace(".", "_")
             .replace(" ", "_")
             .replace("-", "_")
             .replace("/", "_")
             .replace(":", "_")
        )
        new_cols.append(clean)
    return df.toDF(*new_cols)

matches_spark_clean = clean_column_names(matches_spark)

display(matches_spark_clean)

In [0]:
from pyspark.sql.types import NullType

valid_cols = [
    field.name
    for field in matches_spark_clean.schema.fields
    if not isinstance(field.dataType, NullType)
]

matches_spark_no_void = matches_spark_clean.select(*valid_cols)

matches_spark_no_void.printSchema()
display(matches_spark_no_void)

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS futebol")

matches_spark_no_void.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("futebol.bronze_bsa_matches_2026") 

In [0]:
%sql
SELECT * FROM futebol.bronze_bsa_matches_2026 LIMIT 10 